# DocuSign Retention Case Analysis

**Goal:** Identify high-impact opportunities to improve self-serve retention.

**Flow:** Baseline metrics → Where churn happens → Why churn happens → Which segments → Size opportunities

---
## 1. Load Data into DuckDB

In [1]:
import duckdb
import pandas as pd

file = "raw_data.xlsx"

snapshot = pd.read_excel(file, sheet_name="customer base snapshot 1-31-17")
churn    = pd.read_excel(file, sheet_name="churn transactions post snap")
usage    = pd.read_excel(file, sheet_name="usage data 2-1-2018")

# Standardize column names: strip whitespace
for df in [snapshot, churn, usage]:
    df.columns = df.columns.str.strip()

con = duckdb.connect()
con.register("snapshot", snapshot)
con.register("churn",    churn)
con.register("usage",    usage)

print("snapshot:", snapshot.shape)
print("churn   :", churn.shape)
print("usage   :", usage.shape)

print("\n--- snapshot columns ---")
print(snapshot.columns.tolist())
print("\n--- churn columns ---")
print(churn.columns.tolist())
print("\n--- usage columns ---")
print(usage.columns.tolist())

snapshot: (56062, 7)
churn   : (15174, 4)
usage   : (56062, 5)

--- snapshot columns ---
['ACCOUNTID', 'Base MRR in USD', 'SEATS', 'BILLINGCYCLE', 'PURCHASEMONTHYR', 'PLANFAMILY', 'INDUSTRY']

--- churn columns ---
['ACCOUNTID', 'CHURNMONTHYR', 'Churn MRR in USD', 'Churn Type']

--- usage columns ---
['ACCOUNTID', 'LIFETIMESENDS', 'LIFETIMEST', 'LIFETIMETEMPLATECREATION', 'LIFETIMEPOWERFORMS']


---
## 2. Baseline Revenue Metrics
*Slide 1: Size of the business and churn impact*

In [2]:
total_accounts = con.execute("""
    SELECT COUNT(*) AS total_accounts
    FROM snapshot
""").df()
print(total_accounts)

   total_accounts
0           56062


In [3]:
total_mrr = con.execute("""
    SELECT
        ROUND(SUM("Base MRR in USD"), 2)       AS total_mrr,
        ROUND(SUM("Base MRR in USD") * 12, 2)  AS total_arr
    FROM snapshot
""").df()
print(total_mrr)

    total_mrr    total_arr
0  1343947.62  16127371.44


In [4]:
churn_mrr = con.execute("""
    SELECT
        ROUND(SUM("Churn MRR in USD"), 2)       AS churn_mrr,
        ROUND(SUM("Churn MRR in USD") * 12, 2)  AS churn_arr,
        COUNT(*)                                 AS churned_accounts
    FROM churn
""").df()
print(churn_mrr)

   churn_mrr   churn_arr  churned_accounts
0 -338085.86 -4057030.32             15174


In [5]:
# Compute churn rate in Python
t_mrr    = con.execute('SELECT SUM("Base MRR in USD") FROM snapshot').fetchone()[0]
c_mrr    = con.execute('SELECT SUM("Churn MRR in USD") FROM churn').fetchone()[0]
churn_rate = c_mrr / t_mrr

print(f"Total MRR   : ${t_mrr:,.0f}")
print(f"Churn MRR   : ${c_mrr:,.0f}")
print(f"Churn Rate  : {churn_rate:.1%}")

Total MRR   : $1,343,948
Churn MRR   : $-338,086
Churn Rate  : -25.2%


In [6]:
active_passive = con.execute("""
    SELECT
        "Churn Type",
        COUNT(*)                                AS churned_accounts,
        ROUND(SUM("Churn MRR in USD"), 2)       AS churn_mrr,
        ROUND(SUM("Churn MRR in USD") * 12, 2)  AS churn_arr
    FROM churn
    GROUP BY 1
    ORDER BY churn_mrr DESC
""").df()
print(active_passive)

     Churn Type  churned_accounts  churn_mrr   churn_arr
0  PassiveChurn              6329 -144060.47 -1728725.64
1   ActiveChurn              8845 -194025.39 -2328304.68


---
## 3. Build Master Table
*Join snapshot + usage + churn on ACCOUNTID*

In [7]:
con.execute("""
CREATE OR REPLACE TABLE master AS

SELECT
    s.ACCOUNTID,
    s."Base MRR in USD"           AS mrr,
    s."Base MRR in USD" * 12      AS arr,
    s.SEATS,
    s.BILLINGCYCLE,
    s.PLANFAMILY,
    s.INDUSTRY,
    s.PURCHASEMONTHYR,

    u.LIFETIMESENDS,
    u.LIFETIMEST,
    u.LIFETIMETEMPLATECREATION,
    u.LIFETIMEPOWERFORMS,

    CASE WHEN c.ACCOUNTID IS NOT NULL THEN 1 ELSE 0 END AS churned,
    c."Churn Type"                AS churn_type,
    c."Churn MRR in USD"          AS churn_mrr

FROM snapshot s
LEFT JOIN usage u  ON s.ACCOUNTID = u.ACCOUNTID
LEFT JOIN churn c  ON s.ACCOUNTID = c.ACCOUNTID
""")

master_preview = con.execute("""
    SELECT * FROM master LIMIT 5
""").df()
print("Master table shape:", con.execute("SELECT COUNT(*) FROM master").fetchone()[0], "rows")
master_preview

Master table shape: 56062 rows


,ACCOUNTID,mrr,arr,SEATS,BILLINGCYCLE,PLANFAMILY,INDUSTRY,PURCHASEMONTHYR,LIFETIMESENDS,LIFETIMEST,LIFETIMETEMPLATECREATION,LIFETIMEPOWERFORMS,churned,churn_type,churn_mrr
0,DS33930860,420.0,5040.0,14,Annual,Business,Insurance,2015-10,1238,1055,0,0,1,ActiveChurn,-420.0
1,DS23884139,150.0,1800.0,5,Annual,Business,Business Services,2015-11,27969,24953,3,0,1,ActiveChurn,-150.0
2,DS49467295,120.0,1440.0,2,Month,Business,Insurance,2016-09,12707,10753,0,0,1,ActiveChurn,-120.0
3,DS49585374,60.0,720.0,1,Month,Business,Education,2016-08,4433,3925,1,0,1,ActiveChurn,-60.0
4,DS34081096,50.0,600.0,1,Month,Business,Mortgage,2016-02,345,302,0,0,1,ActiveChurn,-50.0


---
## 4. Activation / Usage Analysis
*Hypothesis: Users who never send documents churn significantly more*

*Likely Opportunity #1: Improve activation and early value realization*

In [8]:
usage_churn = con.execute("""
    SELECT
        CASE
            WHEN LIFETIMESENDS = 0          THEN '0'
            WHEN LIFETIMESENDS BETWEEN 1 AND 5  THEN '1-5'
            WHEN LIFETIMESENDS BETWEEN 6 AND 20 THEN '6-20'
            ELSE '20+'
        END AS send_bucket,

        COUNT(*)                                  AS accounts,
        SUM(churned)                              AS churned_accounts,
        ROUND(AVG(churned) * 100, 1)              AS churn_rate_pct,
        ROUND(SUM(churn_mrr), 2)                  AS churn_mrr,
        ROUND(SUM(churn_mrr) * 12, 2)             AS churn_arr

    FROM master
    GROUP BY 1
    ORDER BY
        CASE send_bucket
            WHEN '0'    THEN 1
            WHEN '1-5'  THEN 2
            WHEN '6-20' THEN 3
            ELSE 4
        END
""").df()
print(usage_churn.to_string(index=False))

send_bucket  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
          0      4405            1999.0            45.4  -31902.48  -382829.76
        1-5      5630            2764.0            49.1  -49874.56  -598494.72
       6-20      7166            3625.0            50.6  -69696.63  -836359.56
        20+     38861            6786.0            17.5 -186612.19 -2239346.28


---
## 5. Plan Family Analysis
*Do certain plans have structurally higher churn?*

In [9]:
plan_churn = con.execute("""
    SELECT
        PLANFAMILY,
        COUNT(*)                       AS accounts,
        SUM(churned)                   AS churned_accounts,
        ROUND(AVG(churned) * 100, 1)   AS churn_rate_pct,
        ROUND(SUM(churn_mrr), 2)       AS churn_mrr,
        ROUND(SUM(churn_mrr) * 12, 2)  AS churn_arr
    FROM master
    GROUP BY 1
    ORDER BY churn_arr DESC NULLS LAST
""").df()
print(plan_churn.to_string(index=False))

PLANFAMILY  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
RealEstate     11083            2401.0            21.7  -40713.59  -488563.08
  Personal     18795            6812.0            36.2  -78704.66  -944455.92
  Standard     12609            3026.0            24.0  -94959.73 -1139516.76
  Business     13575            2935.0            21.6 -123707.88 -1484494.56


---
## 6. Billing Cycle Analysis
*Monthly vs Annual — monthly typically churns far more*

In [10]:
billing_churn = con.execute("""
    SELECT
        BILLINGCYCLE,
        COUNT(*)                       AS accounts,
        SUM(churned)                   AS churned_accounts,
        ROUND(AVG(churned) * 100, 1)   AS churn_rate_pct,
        ROUND(SUM(churn_mrr), 2)       AS churn_mrr,
        ROUND(SUM(churn_mrr) * 12, 2)  AS churn_arr
    FROM master
    GROUP BY 1
    ORDER BY churn_arr DESC NULLS LAST
""").df()
print(billing_churn.to_string(index=False))

BILLINGCYCLE  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
       Month     12354            5199.0            42.1 -148363.40 -1780360.80
      Annual     43708            9975.0            22.8 -189722.46 -2276669.52


---
## 7. Industry Churn Analysis
*Which industries drive the most churn ARR?*

In [11]:
industry_churn = con.execute("""
    SELECT
        INDUSTRY,
        COUNT(*)                       AS accounts,
        SUM(churned)                   AS churned_accounts,
        ROUND(AVG(churned) * 100, 1)   AS churn_rate_pct,
        ROUND(SUM(churn_mrr), 2)       AS churn_mrr,
        ROUND(SUM(churn_mrr) * 12, 2)  AS churn_arr
    FROM master
    GROUP BY 1
    ORDER BY churn_arr DESC NULLS LAST
    LIMIT 10
""").df()
print(industry_churn.to_string(index=False))

      INDUSTRY  accounts  churned_accounts  churn_rate_pct  churn_mrr  churn_arr
    Government       234              89.0            38.0   -1607.40  -19288.80
 Life Sciences       408             107.0            26.2   -3070.82  -36849.84
Not for Profit       710             212.0            29.9   -4943.09  -59317.08
    Accounting       952             237.0            24.9   -5714.22  -68570.64
         Legal      1668             331.0            19.8   -7743.63  -92923.56
     Education      1082             353.0            32.6   -8117.55  -97410.60
      Mortgage      1637             302.0            18.4   -8811.13 -105733.56
     Insurance      1735             367.0            21.2   -9993.41 -119920.92
    Healthcare      1731             486.0            28.1  -12558.67 -150704.04
  Construction      2405             545.0            22.7  -12861.61 -154339.32


---
## 8. Value Realization Analysis
*Users get value when agreements are completed (successful transactions)*

*Signals: completed sends (LIFETIMEST), templates, powerforms*

In [12]:
st_churn = con.execute("""
    SELECT
        CASE
            WHEN LIFETIMEST = 0              THEN '0'
            WHEN LIFETIMEST BETWEEN 1 AND 5  THEN '1-5'
            WHEN LIFETIMEST BETWEEN 6 AND 20 THEN '6-20'
            ELSE '20+'
        END AS st_bucket,

        COUNT(*)                       AS accounts,
        SUM(churned)                   AS churned_accounts,
        ROUND(AVG(churned) * 100, 1)   AS churn_rate_pct,
        ROUND(SUM(churn_mrr), 2)       AS churn_mrr,
        ROUND(SUM(churn_mrr) * 12, 2)  AS churn_arr

    FROM master
    GROUP BY 1
    ORDER BY
        CASE st_bucket
            WHEN '0'    THEN 1
            WHEN '1-5'  THEN 2
            WHEN '6-20' THEN 3
            ELSE 4
        END
""").df()
print(st_churn.to_string(index=False))

st_bucket  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
        0      5365            2511.0            46.8  -41715.05  -500580.60
      1-5      6944            3486.0            50.2  -64434.32  -773211.84
     6-20      7269            3455.0            47.5  -71842.82  -862113.84
      20+     36484            5722.0            15.7 -160093.67 -1921124.04


In [13]:
# Template creation usage vs churn
template_churn = con.execute("""
    SELECT
        CASE
            WHEN LIFETIMETEMPLATECREATION = 0 THEN 'No templates'
            ELSE 'Has templates'
        END AS template_bucket,

        COUNT(*)                       AS accounts,
        SUM(churned)                   AS churned_accounts,
        ROUND(AVG(churned) * 100, 1)   AS churn_rate_pct,
        ROUND(SUM(churn_mrr), 2)       AS churn_mrr,
        ROUND(SUM(churn_mrr) * 12, 2)  AS churn_arr

    FROM master
    GROUP BY 1
    ORDER BY 1
""").df()
print(template_churn.to_string(index=False))

template_bucket  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
  Has templates      3092             415.0            13.4  -16276.07  -195312.84
   No templates     52970           14759.0            27.9 -321809.79 -3861717.48


---
## 9. Opportunity Sizing
*Translate insights into recoverable ARR*

In [14]:
# Pull key figures for sizing
opp = con.execute("""
    SELECT
        -- Zero-send segment
        SUM(CASE WHEN LIFETIMESENDS = 0 THEN 1 ELSE 0 END)              AS zero_send_accounts,
        SUM(CASE WHEN LIFETIMESENDS = 0 THEN churned ELSE 0 END)        AS zero_send_churned,
        ROUND(AVG(CASE WHEN LIFETIMESENDS = 0 THEN churned END)*100, 1) AS zero_send_churn_rate,
        ROUND(SUM(CASE WHEN LIFETIMESENDS = 0 THEN churn_mrr ELSE 0 END)*12, 0) AS zero_send_churn_arr,

        -- Overall avg MRR
        ROUND(AVG(mrr), 2)                                              AS avg_mrr
    FROM master
""").df()

print(opp.to_string(index=False))

zero_send_churn_arr = opp['zero_send_churn_arr'].iloc[0]
zero_send_churn_rate = opp['zero_send_churn_rate'].iloc[0]
zero_send_accounts   = opp['zero_send_accounts'].iloc[0]

# If we reduce churn by 20 points in the 0-send segment
reduction_pct = 0.20
arr_saved = zero_send_churn_arr * (reduction_pct / (zero_send_churn_rate / 100))

print(f"\n--- Opportunity #1: Activation ---")
print(f"Zero-send accounts       : {zero_send_accounts:,.0f}")
print(f"Zero-send churn rate     : {zero_send_churn_rate}%")
print(f"Zero-send churn ARR      : ${zero_send_churn_arr:,.0f}")
print(f"If we reduce churn by 20pp -> ARR saved: ${arr_saved:,.0f}")

 zero_send_accounts  zero_send_churned  zero_send_churn_rate  zero_send_churn_arr  avg_mrr
             4405.0             1999.0                  45.4            -382830.0    23.97

--- Opportunity #1: Activation ---
Zero-send accounts       : 4,405
Zero-send churn rate     : 45.4%
Zero-send churn ARR      : $-382,830
If we reduce churn by 20pp -> ARR saved: $-168,648


In [15]:
# Opportunity #2: Passive churn recovery
passive = con.execute("""
    SELECT
        "Churn Type",
        COUNT(*)                                  AS accounts,
        ROUND(SUM("Churn MRR in USD") * 12, 0)   AS churn_arr
    FROM churn
    GROUP BY 1
""").df()
print(passive.to_string(index=False))

passive_arr = con.execute("""
    SELECT ROUND(SUM("Churn MRR in USD") * 12, 0) AS passive_arr
    FROM churn
    WHERE "Churn Type" ILIKE '%passive%'
""").fetchone()[0]

if passive_arr:
    print(f"\n--- Opportunity #2: Passive Churn Recovery ---")
    print(f"Passive churn ARR        : ${passive_arr:,.0f}")
    # Recovery scenario: dunning / billing retry recovers 30%
    recovery_rate = 0.30
    print(f"Recovery rate (30%)      : ${passive_arr * recovery_rate:,.0f} ARR")

  Churn Type  accounts  churn_arr
PassiveChurn      6329 -1728726.0
 ActiveChurn      8845 -2328305.0

--- Opportunity #2: Passive Churn Recovery ---
Passive churn ARR        : $-1,728,726
Recovery rate (30%)      : $-518,618 ARR


---
## 10. Final Deliverable Tables
*Clean summary tables for slides*

In [16]:
# Table 1: Baseline churn metrics
t_accounts = con.execute("SELECT COUNT(*) FROM snapshot").fetchone()[0]
t_mrr_val  = con.execute('SELECT SUM("Base MRR in USD") FROM snapshot').fetchone()[0]
c_mrr_val  = con.execute('SELECT SUM("Churn MRR in USD") FROM churn').fetchone()[0]
c_accounts = con.execute("SELECT COUNT(*) FROM churn").fetchone()[0]

table1 = pd.DataFrame({
    'Metric': ['Total Accounts', 'Total MRR', 'Total ARR', 'Churned Accounts', 'Churn MRR', 'Churn ARR', 'MRR Churn Rate'],
    'Value': [
        f"{t_accounts:,}",
        f"${t_mrr_val:,.0f}",
        f"${t_mrr_val*12:,.0f}",
        f"{c_accounts:,}",
        f"${c_mrr_val:,.0f}",
        f"${c_mrr_val*12:,.0f}",
        f"{c_mrr_val/t_mrr_val:.1%}"
    ]
})
print("=== TABLE 1: Baseline Churn Metrics ===")
print(table1.to_string(index=False))

=== TABLE 1: Baseline Churn Metrics ===
          Metric       Value
  Total Accounts      56,062
       Total MRR  $1,343,948
       Total ARR $16,127,371
Churned Accounts      15,174
       Churn MRR   $-338,086
       Churn ARR $-4,057,030
  MRR Churn Rate      -25.2%


In [17]:
# Table 2: Active vs Passive churn
print("=== TABLE 2: Active vs Passive Churn ===")
print(active_passive.to_string(index=False))

=== TABLE 2: Active vs Passive Churn ===
  Churn Type  churned_accounts  churn_mrr   churn_arr
PassiveChurn              6329 -144060.47 -1728725.64
 ActiveChurn              8845 -194025.39 -2328304.68


In [18]:
# Table 3: Usage (sends) vs Churn
print("=== TABLE 3: Usage (Lifetime Sends) vs Churn ===")
print(usage_churn.to_string(index=False))

=== TABLE 3: Usage (Lifetime Sends) vs Churn ===
send_bucket  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
          0      4405            1999.0            45.4  -31902.48  -382829.76
        1-5      5630            2764.0            49.1  -49874.56  -598494.72
       6-20      7166            3625.0            50.6  -69696.63  -836359.56
        20+     38861            6786.0            17.5 -186612.19 -2239346.28


In [19]:
# Table 4: Plan churn
print("=== TABLE 4: Plan Family Churn ===")
print(plan_churn.to_string(index=False))

=== TABLE 4: Plan Family Churn ===
PLANFAMILY  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
RealEstate     11083            2401.0            21.7  -40713.59  -488563.08
  Personal     18795            6812.0            36.2  -78704.66  -944455.92
  Standard     12609            3026.0            24.0  -94959.73 -1139516.76
  Business     13575            2935.0            21.6 -123707.88 -1484494.56


In [20]:
# Table 5: Billing cycle churn
print("=== TABLE 5: Billing Cycle Churn ===")
print(billing_churn.to_string(index=False))

=== TABLE 5: Billing Cycle Churn ===
BILLINGCYCLE  accounts  churned_accounts  churn_rate_pct  churn_mrr   churn_arr
       Month     12354            5199.0            42.1 -148363.40 -1780360.80
      Annual     43708            9975.0            22.8 -189722.46 -2276669.52


In [21]:
# Table 6: Industry churn (top 10)
print("=== TABLE 6: Industry Churn (Top 10 by ARR) ===")
print(industry_churn.to_string(index=False))

=== TABLE 6: Industry Churn (Top 10 by ARR) ===
      INDUSTRY  accounts  churned_accounts  churn_rate_pct  churn_mrr  churn_arr
    Government       234              89.0            38.0   -1607.40  -19288.80
 Life Sciences       408             107.0            26.2   -3070.82  -36849.84
Not for Profit       710             212.0            29.9   -4943.09  -59317.08
    Accounting       952             237.0            24.9   -5714.22  -68570.64
         Legal      1668             331.0            19.8   -7743.63  -92923.56
     Education      1082             353.0            32.6   -8117.55  -97410.60
      Mortgage      1637             302.0            18.4   -8811.13 -105733.56
     Insurance      1735             367.0            21.2   -9993.41 -119920.92
    Healthcare      1731             486.0            28.1  -12558.67 -150704.04
  Construction      2405             545.0            22.7  -12861.61 -154339.32
